# Lab 09: Assignment 2 tutorial

How would you store a matrix with only three nonzero diagonals?

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh

## Three diagonals

`diags` takes the entries and their offsets. Offset 0 is the main diagonal;
offsets $-1$ and $+1$ sit below and above it.

We'll use a small matrix we can inspect. How many entries belong on each diagonal?

In [ ]:
A = diags(
    [np.array([-0.2, -0.3, -0.3, -0.2]),
     np.array([3.0, 4.0, 5.0, 4.0, 3.0]),
     np.array([-0.2, -0.3, -0.3, -0.2])],
    offsets=[-1, 0, 1], format="csc",
)
print(A.toarray())

fig, ax = plt.subplots(figsize=(4, 3), layout="constrained")
ax.spy(A, markersize=12)
ax.set(xlabel="Column", ylabel="Row")
plt.show()

## Find two eigenvalues

Shift-invert looks near the shift $\sigma$. For this positive definite matrix, use
$\sigma=0$. `which="LM"` selects the largest magnitudes after the transformation.

Each column of `vectors` is an eigenvector. Which eigenvalue goes with it?

In [ ]:
values, vectors = eigsh(A, k=2, sigma=0.0, which="LM", v0=np.arange(1.0, 6.0))
order = np.argsort(values)
values = values[order]
vectors = vectors[:, order]
print("Eigenvalues:", values)
print("Eigenvector shape:", vectors.shape)

### Check a returned pair

Calculate $Av_j-E_jv_j$ for each column. What would you expect?

Now multiply one eigenvector by $-1$ and repeat the check. Did the state change?

## Which norm?

`eigsh` returns vectors with $\sum_i|v_i|^2=1$. On a position grid we need

$$
\langle a|b\rangle\approx\Delta x\sum_i a_i^*b_i.
$$

`np.vdot` conjugates its first argument. Predict these three results.

In [ ]:
v = np.array([1.0, 1.0j, -1.0])
spacing = 0.2
print("No conjugation:", v @ v)
print("With conjugation:", np.vdot(v, v))
print("Grid sum:", spacing * np.vdot(v, v))

### Try it

Normalize `v` with the grid sum and check the result.
What goes wrong if you divide by the norm squared instead of its square root?

For a normalized state, how would you calculate $\langle x\rangle$ on the grid?

## Build a Hamiltonian

For the harmonic oscillator with $m=\hbar=1$,

$$
V(x)=\frac12\omega^2x^2,\qquad E_n=\left(n+\frac12\right)\omega.
$$

Try $\omega=2$. Write out the first and last rows of the centered three-point
Hamiltonian, then build its three diagonals. Retain the endpoint unknowns;
their missing neighbors outside the box are zero.

Find the lowest two energies. Compare with 1 and 3, normalize the eigenvectors
using your grid spacing, and check their overlap.

Use the double-well potential for Assignment 2.

## Check the grid

A small matrix residual checks the eigensolver. Does it check the approximation
to the differential equation?

Halve the spacing while keeping the box fixed. Then enlarge the box while
keeping the spacing fixed. Rebuild the Hamiltonian each time.

Which entries change in each test? Compare the two lowest energies and their gap.

### Your turn: the double well

Use the potential and grids in [Assignment 2, part B](https://jduarte.physics.ucsd.edu/phys142/assignment2.pdf).

1. Check $\alpha=0.4,0.6$ against Table I from Lecture 11. The paper's
   parameter is $\lambda=\alpha/4$.
2. At your assigned $\alpha$, implement the three grid tests in B(ii).
   How many digits of the gap can you justify?
3. Normalize and plot the two eigenfunctions. Choose the ground state positive
   and the first excited state's right lobe positive. Check their parities.

Keep the original-grid results for the later parts and `results.json`.

## Rows and columns

Before building a propagator, check how broadcasting makes a matrix.
What shape does each array below have? What will entry `[2, 3]` be?

In [ ]:
row = np.arange(3)[:, None]
column = np.arange(4)[None, :]
labels = 10 * row + column
print("Shapes:", row.shape, column.shape, labels.shape)
print(labels)

## One matrix step

For `M @ a`, each row gives one output component. Work out both components
before running the cell. Does matrix multiplication preserve the norm here?

In [ ]:
M = np.array([[1, 1j], [2, 1]], dtype=complex)
a = np.array([1, 0.5j])
b = M @ a
print("Before:", a)
print("After: ", b)
print("Squared norms:", np.vdot(a, a).real, np.vdot(b, b).real)

## Your turn: propagation

Build the midpoint kernel from part A. Rows are destinations; columns are
sources. What shape should the matrix have? Where does the factor of $\Delta x$
enter the grid integral?

Start with the right-well Gaussian. Implement one step, normalize it with the
grid sum, and check the norm. Then write the time loop, including $t=0$ in your
stored results.

Use B's gap to estimate a full slow period. Plot $\langle x\rangle(t)$ for the
trip to the left well and back.

## Read the motion

Mark the first major minimum of your curve. Give its time and value.
If there are faster fluctuations, mark one too.

Would a global `argmin` select the first major minimum? Choose a time window
from your plot and explain the choice.

The first zero crossing is not the tunneling time. Compare your measured time
with $\pi/\Delta E$ after locating the minimum.

## Energy of a trial state

$$
E[\chi]=\frac{\langle\chi|H|\chi\rangle}{\langle\chi|\chi\rangle}.
$$

Use our small matrix `A` again. The vector below is not normalized.
Form the ratio from the two printed values. Then multiply `trial` by 3 and
repeat. Why does the energy stay the same?

In [ ]:
trial = np.ones(5)
print("Numerator:", np.vdot(trial, A @ trial).real)
print("Denominator:", np.vdot(trial, trial).real)

## Errors in a small gap

Suppose the exact energies are 1.00 and 1.02, and the estimates are 1.01 and 1.04.
Calculate the relative error in each energy and in the gap.

Use the signed errors to explain the gap error. Does a small relative error in
each energy guarantee a small relative error in their difference?

### Your turn: harmonic approximations

Construct $\psi_\mathrm{S}$ and $\psi_\mathrm{A}$ from part C. Evaluate their
energies using B's Hamiltonian and grid sums. Keep the denominators: the local
Gaussians overlap.

Compare the energies and their gap with B. Account for the gap error using the
signed energy errors.

Normalize $\psi_\mathrm{S}$ and overlay $|\psi_\mathrm{S}|$ and $|\phi_0|$ on a log
scale. Mark a discrepancy in the barrier region. How does it connect to your
energy comparison?

## A phase factor

In units with $\hbar=1$, an energy eigenstate picks up the phase $e^{-iEt}$.
What happens to its probability density?

In [ ]:
energy = 0.7
sample_times = np.array([0.0, 1.0, 2.0])
phase = np.exp(-1j * energy * sample_times)
print("Phase:", phase)
print("Magnitude squared:", np.abs(phase)**2)

## Your turn: two eigenstates

For part D, use $(\phi_0+\phi_1)/\sqrt{2}$ from B and $T=2\pi/\Delta E$.

1. Give each term its time-dependent phase. Expand $|\psi(x,t)|^2$ and derive
   $\langle x\rangle(t)$. Which terms vanish by parity? Where does normalization enter?
2. Implement that evolution and reuse your propagator with the same initial state.
   Keep this run separate from A's Gaussian run.
3. Compare both means at every step over one full period. Find the largest
   difference for $t\leq T$, and report its time and both means there.
4. Overlay the densities at the nearest steps to $0,T/2,T$. Label the actual times.

Use your plots to propose a numerical cause for a discrepancy. What check would
test it, and what would you expect? Part D asks for the proposal; no extra run is required.

Can two calculations both have norm 1 and still disagree about the motion?

## Save a small result

NumPy arrays need `.tolist()` before writing them as JSON. Write these sample
readings, then open the file again. What Python type comes back for `readings`?

In [ ]:
readings = np.array([2.0, 2.5, 3.0])
practice = {"label": "practice readings", "readings": readings.tolist()}
practice_path = Path("lab09_practice.json")
practice_path.write_text(json.dumps(practice, indent=2, allow_nan=False))
loaded = json.loads(practice_path.read_text())
print(loaded)
print(type(loaded["readings"]))

## Assignment 2 export times

Select steps $n=0,64,128,\ldots,2048$: 33 values, ending at $16\pi$.
Run A through at least this time, and longer if needed for a full slow oscillation.
Evaluate D's eigenstate formula at all 33 times too.

If you store the initial state, which array entry corresponds to step 64?

### `results.json`

| Fields | Values |
| :--- | :--- |
| `tsn`, `alpha` | Nine-digit TSN as text; $\alpha=0.3+d/5000$ from its last three digits |
| `x_mean_pi`, `t_tunnel` | A's Gaussian run: 33 means and the measured tunneling time |
| `E0`, `E1`, `phi0`, `phi1` | B's original-grid energies and signed, normalized eigenvectors; 601 values per vector |
| `E_S`, `E_A` | C's two energy estimates |
| `x_mean_eig` | D's eigenstate means at the 33 export times |

Use the original grid for these fields. D's path-integral curve goes in the report,
not in `x_mean_pi`. `allow_nan=False` catches nonfinite values when saving.

Assignment 2 is due Monday, November 2.

### Bonus

For E, compare the measured Gaussian tunneling time with $\pi/\Delta E$ at
$\alpha=0.30,0.35,0.40,0.45,0.50$. Calculate the fraction of the normalized
Gaussian in the lowest two states. Does it help explain the timing differences
and faster oscillations?

If you do E, add `delta_E_list` and `t_tunnel_list` to `results.json` in that order
of $\alpha$. Omit them if you skip it.